In [49]:
from dotenv import load_dotenv

load_dotenv()

True

In [50]:
from langchain_ollama import ChatOllama

model = ChatOllama(model = "minimax-m3:cloud", temperature = 0)

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def calorie_calc(food:str) -> str:

    """Look for amount of calories in each food ingredient"""

    calories = {
    # Fruits
    "apple": 95,              # 1 medium
    "banana": 105,            # 1 medium
    "orange": 62,             # 1 medium
    "mango": 150,             # 1 medium
    "pear": 101,              # 1 medium
    "peach": 59,              # 1 medium
    "grapes": 104,            # 1 cup
    "strawberries": 49,       # 1 cup
    "watermelon": 46,         # 1 cup
    "pineapple": 82,          # 1 cup

    # Vegetables
    "potato": 161,            # 1 medium
    "sweet potato": 112,      # 1 medium
    "carrot": 25,             # 1 medium
    "broccoli": 55,           # 1 cup cooked
    "spinach": 41,            # 1 cup cooked
    "tomato": 22,             # 1 medium
    "cucumber": 16,            # 1/2 cucumber
    "corn": 143,              # 1 medium ear
    "peas": 134,              # 1 cup cooked

    # Protein / meat
    "egg": 78,                # 1 large
    "chicken breast": 165,    # 100 g cooked
    "chicken thigh": 209,     # 100 g cooked
    "beef": 250,              # 100 g cooked
    "mutton": 250,            # 100 g cooked
    "lamb": 250,              # 100 g cooked
    "fish": 206,              # 100 g cooked
    "salmon": 206,            # 100 g cooked
    "tuna": 132,              # 100 g
    "shrimp": 99,             # 100 g cooked

    # Grains / bread
    "white rice": 205,        # 1 cup cooked
    "brown rice": 218,        # 1 cup cooked
    "rice": 206,              # 1 cup cooked
    "bread": 80,              # 1 slice
    "whole wheat bread": 82,  # 1 slice
    "roti": 120,              # 1 medium
    "chapati": 120,           # 1 medium
    "naan": 260,              # 1 medium
    "oatmeal": 158,           # 1 cup cooked
    "pasta": 221,             # 1 cup cooked

    # Dairy
    "milk": 103,              # 1 cup
    "whole milk": 149,        # 1 cup
    "yogurt": 150,             # 1 cup
    "cheese": 113,             # ~28 g
    "butter": 102,             # 1 tbsp

    # Fast food / prepared foods
    "pizza": 285,              # 1 slice
    "burger": 354,             # 1 burger
    "french fries": 365,       # 1 medium serving
    "hot dog": 151,            # 1
    "sandwich": 300,           # 1 average
    "fried chicken": 320,      # 1 piece
    "chicken nuggets": 280,    # ~6 pieces

    # Snacks / nuts
    "almonds": 164,            # 1 oz
    "peanuts": 166,            # 1 oz
    "walnuts": 185,            # 1 oz
    "cashews": 157,            # 1 oz
    "popcorn": 93,             # 3 cups air-popped
    "potato chips": 152,       # ~1 oz

    # Desserts / sweets
    "chocolate": 170,          # ~1 oz
    "ice cream": 207,          # 1/2 cup
    "cake": 350,               # 1 slice
    "cookie": 140,             # 1 large
    "donut": 250,              # 1
}

    food = food.lower().strip()

    if food in calories:
        return f"{food} contains approximately {calories[food]} calories."

    return f"Sorry, I don't have calorie information for {food}."
    

In [52]:
system_prompt = """

You are a personal chef of a heart patient. The user will give you a list of ingredients they have left over in their house,

and an image of open fridge. Look for the additional ingredients in the fridge. List down those ingredients.

Look for amount of calories in each ingredient using calorie calc tool.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [57]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[web_search, calorie_calc],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [58]:
#Image Input
from ipywidgets import FileUpload
from IPython.display import display
import base64

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)





FileUpload(value=(), accept='.png', description='Upload')

In [59]:
# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [60]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

TextMsg = HumanMessage(content= [
    {"type":"text", "text":"I have some leftover milk, cheese and pasta. What can I make? Calculate the amount of calories in each recipe"},
    {"type":"image","base64":img_b64,"mime_type":"image/png"}
])
response = agent.invoke(
    {"messages": [TextMsg]},
    config
)

print(response['messages'][-1].content)

Here's what I found for you! 📋

## 🚨 Fridge Status
Your fridge is **completely empty** — no additional ingredients available. So we'll work with just your 3 leftovers: **milk, cheese, and pasta**.

## 🔥 Calorie Information (from available tool data)
| Ingredient | Calories |
|------------|----------|
| Whole milk (1 cup) | **~149 cal** |
| Cheddar cheese (1 oz) | ~115 cal (standard nutrition value) |
| Cooked pasta (1 cup) | ~220 cal (standard nutrition value) |

*Note: The calorie tool only returned data for whole milk. Cheese and pasta values are based on standard nutrition data.*

## ❤️ Heart-Healthy Recipe Suggestions

Given your heart condition, I recommend **lighter versions** that use reduced-fat cheese and skim milk instead of whole milk to lower saturated fat. Here are the top options:

### 1. **Healthy Baked Mac and Cheese with Spinach** ⭐ (Top Pick)
*A lighter comfort-food classic from Skinnytaste — ~300 cal per serving*
- Uses reduced-fat cheese + skim milk (instead of whol

In [61]:
from pprint import pprint

pprint(response["messages"][-1].tool_calls)

[]


In [62]:
print(response['messages'][1].tool_calls)

[{'name': 'calorie_calc', 'args': {'food': 'cheese (1 oz, cheddar)'}, 'id': 'a2f135be-f92a-4143-9b12-a507f6bf6d2e', 'type': 'tool_call'}, {'name': 'calorie_calc', 'args': {'food': 'pasta (1 cup cooked)'}, 'id': '2db194a9-f8af-40da-8c5d-fb5896705997', 'type': 'tool_call'}, {'name': 'web_search', 'args': {'query': 'heart healthy recipes with milk cheese pasta for cardiac patients'}, 'id': '8bcd7ed8-8c4a-4711-9878-813c71312092', 'type': 'tool_call'}, {'name': 'calorie_calc', 'args': {'food': 'milk (1 cup, whole)'}, 'id': 'f8a82f09-9551-455c-85ce-b39f9b005255', 'type': 'tool_call'}]
